In [13]:
import pandas as pd
import numpy as np

df_test = pd.read_csv('../data/test.csv')
df_train = pd.read_csv('../data/train.csv')

df_summary = pd.DataFrame({'dtype': df_train.dtypes, 'nulls': df_train.isna().sum(), 'nununique': df_train.nunique(), 'sample_1': df_train.iloc[0], 'sample_2': df_train.iloc[1], 'sample_3': df_train.iloc[2]})
df_summary



,dtype,nulls,nununique,sample_1,sample_2,sample_3
Id,int64,0,1460,1,2,3
MSSubClass,int64,0,15,60,20,60
MSZoning,str,0,5,RL,RL,RL
LotFrontage,float64,259,110,65.0,80.0,68.0
LotArea,int64,0,1073,8450,9600,11250
...,...,...,...,...,...,...
MoSold,int64,0,12,2,5,9
YrSold,int64,0,5,2008,2007,2008
SaleType,str,0,9,WD,WD,WD
SaleCondition,str,0,6,Normal,Normal,Normal


In [14]:
for df in [df_train, df_test]:
    df['Total_SF'] = df['GrLivArea'] + df['TotalBsmtSF']
    df['Total_Bathrooms'] = df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'] + df['FullBath'] + 0.5 * df['HalfBath']
    df['AgeAtSale'] = df['YrSold'] - df['YearBuilt']

In [15]:
num_cols = df_train.select_dtypes(include='number').columns.tolist()

cat_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

ord_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageCond', 'GarageQual', 'PoolQC', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

cat_cols = [col for col in cat_cols if col not in ord_cols]

ord_cols = [col for col in ord_cols if col in df_train.columns]

num_cols = [col for col in num_cols if col != 'SalePrice' and col != 'Id']

C:\Users\yazid\AppData\Local\Temp\ipykernel_11784\2204705351.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


In [16]:

quality_order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
quality_order_none = ['None'] + quality_order
exposure_none_order = ['None', 'No', 'Mn', 'Av', 'Gd']
finish_none_order = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
pool_none_order = ['None', 'Fa', 'Gd', 'Ex']

regular_ord_cols = ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']
none_ord_cols = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'GarageCond', 'GarageQual', 'PoolQC', 'FireplaceQu']
print(cat_cols)
none_cat_cols = ['Alley', 'MasVnrType', 'GarageType', 'GarageFinish', 'Fence', 'MiscFeature']
regular_cat_cols = [col for col in cat_cols if col not in none_cat_cols]


['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [17]:

ordinal_categories = {
    'ExterQual': quality_order,
    'ExterCond': quality_order,
    'BsmtQual': quality_order_none,
    'BsmtCond': quality_order_none,
    'HeatingQC': quality_order,
    'KitchenQual': quality_order,
    'FireplaceQu': quality_order_none,
    'GarageCond': quality_order_none,
    'GarageQual': quality_order_none,
    'PoolQC': pool_none_order,
    'BsmtExposure': exposure_none_order,
    'BsmtFinType1': finish_none_order,
    'BsmtFinType2': finish_none_order
}

regular_categories = [ordinal_categories[col] for col in regular_ord_cols]
none_categories = [ordinal_categories[col] for col in none_ord_cols]



In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median'))])
regular_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
none_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
regular_ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('ordinal', OrdinalEncoder(categories=regular_categories, handle_unknown='use_encoded_value', unknown_value=-1))])
none_ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('ordinal', OrdinalEncoder(categories=none_categories, handle_unknown='use_encoded_value', unknown_value=-1))])

param_distributions = {
    'model__n_estimators': [100, 300, 600, 1000],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__max_depth': [2,3,4,5,6],
    'model__min_child_weight': [1,3,5],
    'model__subsample': [0.8,0.9,1.0]
}
preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols), ('regular_cat', regular_cat_pipeline, regular_cat_cols), ('none_cat', none_cat_pipeline, none_cat_cols), ('regular_ord', regular_ord_pipeline, regular_ord_cols), ('none_ord', none_ord_pipeline, none_ord_cols)])

model1 = XGBRegressor(learning_rate=0.01, n_estimators=100, random_state=42, max_depth=6)

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model1)])

X_train = df_train.drop(columns=['SalePrice', 'Id'])
X_test = df_test.drop(columns=['Id'])
y_train = np.log1p(df_train['SalePrice'])
test_id = df_test['Id']

In [19]:
from sklearn.dummy import DummyRegressor
from xgboost import cv
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

baseline = DummyRegressor(strategy='mean')

baseline_scores = cross_val_score(
    baseline, 
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error'
)

baseline_rmsle = np.sqrt(-baseline_scores.mean())

In [20]:

msle = cross_val_score(full_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
rmsle = np.sqrt(-msle.mean())

print(baseline_rmsle, rmsle)

0.3995729236866519 0.21508338363751625


In [21]:
search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    refit=False,
    error_score='raise'
)

search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_)

In [22]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__subsample,param_model__n_estimators,param_model__min_child_weight,param_model__max_depth,param_model__learning_rate,params,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,2.507628,0.043955,0.065199,0.007333,0.9,300,3,6,0.03,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.018439,0.006086,16,-9.909006e-04,-9.749353e-04,-8.919312e-04,-1.017417e-03,-9.832783e-04,-9.716925e-04,4.234693e-05
1,1.039774,0.040913,0.080887,0.010988,1.0,100,1,5,0.03,"{'model__subsample': 1.0, 'model__n_estimators...",...,-0.021103,0.004351,21,-6.336726e-03,-6.448748e-03,-6.210128e-03,-6.587251e-03,-6.695492e-03,-6.455669e-03,1.728091e-04
2,1.772054,0.140723,0.060980,0.003376,0.8,600,3,3,0.01,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.018097,0.004754,14,-8.767179e-03,-9.064181e-03,-8.125191e-03,-8.997590e-03,-9.079574e-03,-8.806743e-03,3.586674e-04
3,1.140089,0.019233,0.057751,0.005901,0.8,600,3,2,0.03,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.016799,0.004723,2,-7.579662e-03,-7.783049e-03,-7.155479e-03,-7.848028e-03,-7.729665e-03,-7.619177e-03,2.481984e-04
4,4.006186,0.319528,0.110360,0.010737,0.8,1000,3,4,0.05,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.016435,0.005115,1,-1.998860e-04,-2.100963e-04,-1.959993e-04,-2.380887e-04,-2.279665e-04,-2.144073e-04,1.620475e-05
5,0.922228,0.028372,0.071037,0.012047,1.0,300,5,3,0.01,"{'model__subsample': 1.0, 'model__n_estimators...",...,-0.022787,0.003991,26,-1.532046e-02,-1.607415e-02,-1.435595e-02,-1.544305e-02,-1.616355e-02,-1.547143e-02,6.496457e-04
6,1.138592,0.199492,0.096887,0.037045,0.8,300,5,3,0.01,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.022108,0.003657,24,-1.530848e-02,-1.587857e-02,-1.426633e-02,-1.537029e-02,-1.605089e-02,-1.537491e-02,6.235400e-04
7,1.900107,0.209477,0.093493,0.016088,0.9,100,1,6,0.01,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.045849,0.006642,29,-3.238836e-02,-3.380608e-02,-3.428591e-02,-3.295343e-02,-3.495554e-02,-3.367786e-02,9.171948e-04
8,2.454216,0.129326,0.093145,0.011109,0.9,300,1,5,0.03,"{'model__subsample': 0.9, 'model__n_estimators...",...,-0.017692,0.004916,12,-1.821434e-03,-1.815324e-03,-1.749454e-03,-1.868066e-03,-1.885393e-03,-1.827934e-03,4.748169e-05
9,1.512649,0.085249,0.088953,0.009529,0.8,600,5,2,0.05,"{'model__subsample': 0.8, 'model__n_estimators...",...,-0.017307,0.004890,9,-5.957481e-03,-6.141323e-03,-5.628816e-03,-6.470993e-03,-6.241024e-03,-6.087928e-03,2.831355e-04


In [23]:
results['train_rmsle'] = np.sqrt(-results['mean_train_score'])

results['validation_rmsle'] = np.sqrt(
    -results['mean_test_score']
)

results['overfit_gap'] = (
    results['validation_rmsle'] - results['train_rmsle']
)

max_gap = 0.05

accepted_results = results[results['overfit_gap'] <= max_gap]

best_index = accepted_results['validation_rmsle'].idxmin()

best_params = results.loc[best_index, 'params']

In [24]:
from sklearn.base import clone

final_pipeline = clone(full_pipeline)
final_pipeline.set_params(**best_params)
final_pipeline.fit(X_train, y_train)

selected_scores = -cross_val_score(
    final_pipeline, 
    X_train, 
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error'
)

print("Selected RMSLE:", np.sqrt(selected_scores.mean()))

Selected RMSLE: 0.12946675968144394
